### Análise sobre área fora do mapa

Conecta-se na base para buscar os registros

In [1]:
import os, sys
from pyspark.sql import SparkSession

sys.path.append(os.path.join(os.getcwd(), "/home/jovyan/work/src/pyspark_app"))

import session

spark = SparkSession.builder.getOrCreate()
url = session.jdbc_url()
props = session.jdbc_props()

table_points = "gps.points"

df = spark.read.jdbc(
    url=url,
    table=table_points,
    properties=props
)

df_points = df.toPandas()

Seleciona quais as rotas têm pontos fora do mapa, agrupando-as

In [2]:
df_points[df_points['is_inside_area'] == False].groupby('trip_name').size().reset_index(name='count')

In [3]:
df_points_rota_2 = df_points[(~df_points['is_inside_area']) & (df_points['trip_name'] == '1140 ROTA 2')]

df_points_rota_73 = df_points[(~df_points['is_inside_area']) & (df_points['trip_name'] == '11:45 ROTA 73')]


In [6]:
import folium

# Suponha que você tenha dois DataFrames:
# df1 = pontos fora da área
# df2 = pontos dentro da área
df1 = df_points_rota_2
df2 = df_points_rota_73  # outro DataFrame no mesmo formato (lat/lon)

# Cria o mapa centralizado na média dos pontos
m = folium.Map(
    location=[df1['lat'].mean(), df1['lon'].mean()],
    zoom_start=14,
    tiles='OpenStreetMap'
)

fg_out = folium.FeatureGroup(name="Pontos fora da área")
for _, row in df1.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=4,
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.8
    ).add_to(fg_out)

fg_in = folium.FeatureGroup(name="Pontos dentro da área")
for _, row in df2.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=4,
        color='blue',
        fill=True,
        fill_color='blue',
        fill_opacity=0.8
    ).add_to(fg_in)

# Adiciona as camadas ao mapa
fg_out.add_to(m)
fg_in.add_to(m)

# Adiciona controle de camadas (para ligar/desligar)
folium.LayerControl().add_to(m)

# Mostra o mapa
m
